##### Proyecto PAPIME PE110923 DESARROLLO DE UN LABORATORIO DE ROBOTICA REMOTO PARA REALIZAR PRACTICAS DE PROGRAMACION DE ALGORITMOS DE PLANEACION Y DE NAVEGACION EN BANCOS DE PRUEBA FISICOS

# Este material fue desarrollado con el apoyo del PAPIME PE110923 de la UNAM
## Academia de Robótica

## Autores

- M.I. Erik Peña Medina
- Dr. Víctor Javier Gonzáles Villela


# Práctica 6 Generación de un Entorno de Simulacion en Gazebo

## Objetivo

El objetivo de esta práctica es que el alumno tenga la capacidad de generar entornos de simulación en Gazebo con el fin de que puedan realizar experimentos y pruebas virtuales. 

### Metas 

- Que el alumno aprenda a editar archivos Simulation Data File (.world) en plataformna de Gazebo.
- Crear un paquete *bringup* en el cual se ejecute una simulación donce un robot determinado realice alguna tarea.
  


### Contribución al perfil del egresado

La siguiente práctica contribuye en los siguientes puntos al perfil del egresado:

#### Aptitudes y habilidades

- Para modelar, simular e interpretar el comportamiento de los sistemas mecatrónicos.
- Para desarrollar, operar y mantener procesos productivos que impliquen la transformación de materia, energía e información.
- Para diseñar, construir, operar y mantener los sistemas mecatrónicos y sus componentes.

#### Actitudes

- Ser creativo e innovador.
- Tener confianza en su preparación académica.
- Comprometido con su actualización, superación y competencia profesional.

#### De tipo social

- Promover el cambio en la mentalidad frente a la competitividad internacional.



## Rúbrica de evaluación

La evaluación de la práctica contará de los siguientes puntos:

| Elemento | Porcentaje |
| ------:| ----------- |
| ***Previo*** | 25% | 
| **Desarrollo** | 25% |
| **Resultado**  | 25% |
| **Conclusiones** | 25% |


Los siguientes elementos se evaluaran con base en los siguientes criterios:

| Elemento | Malo | Regular | Bueno | Muy bueno| 
| ------:| ------ | --------| ------| --------  |
| ***Previo*** | El previo no contiene los elemtos solocitados (0%)| El previo contiene parcialmente la infomración solocitada (10%) |  El previo contiene la infomración solocitada sin referencias (20%) | El previo contiene la información solicitada con referencias (25%) |
| **Desarrollo** | El desarrollo de la práctica no fue realizada (0%) | El desarrollo de la práctica fue pacialmente realizado (10%) | La práctica fue realizada pero no fue realizada correctamente (20%) | La práctica fue realizada en su totalidad correctamente (25%) |
| **Resultado**  | No se entregarón los resultados de la práctica (0%)| Se entregaron pracialmente los resultados (10%) | Se entregaron los resultados sin descripción (20%) | Se entregaron los resultados de la práctica y son interpetrables(25%) |
| **Conclusiones** | No se entragaron conclusiones (0%) | Las conclusciones no están relacionadas con el objetivo de la práctica (5%) | Las conclusciones se relacionan con el objetivo de práctica (10%) | Las concluscions se relacionana con el objetivo de la práctica y se basan en los resultados obtenidos (25%) | 


## Desarrollo de la práctica 

Para el desarrollo de la práctica se deben realizar las siguientes activiadades:

1. Explicar las relaciones de los archivos xacros.

La relación que tienen estos archivos es la modularidad del diseño. Lo que se busca es que el robot escara se divida en tres diferentes módulos que interactúan entre sí, evitando tener un código monolítico:

- El modelo mecánico (scara_urdf.xacro): Este actúa como el esqueleto del ensambre. Aquí únicamente definimos la geometríam las propiedades de masa y la cinemática.
- Los controladores (scara_gz2.xacro): Aquí se relaciona con el modelo matemático tomando los nombres de las articulaciones y les "inyecta" las propiedades de control para Gazebo mediante etiquetas de inclusión. Básicamente estamos definiendo los parámetros de control PID para que el simulador pueda mover los motores del modelo
- La percepción (camera_sensor.xacro): Funciona como un módulo períférico. Este se relaciona acoplándose de forma rígida a nuestro modelo mecánico base y añadiendo las etiquetas de sumulación de Gazebo para poder generar los datos de visión artificial. 

2. Decribir las partes del archivo xacro de la configuración de las juntas del robot.

En el archivo podemos observar que el robot está contruido por dos elementos fundamentales:

- Los eslabones (link): El robot cuenta con un base_link (que es su base fija) y tres eslabones móviles, además de un punto de herramienta P. Cada uno de los eslabones está definido por tres etiquetas internas:
  
    - visual: Define el aspecto gráfico.
    - collision: Define el volumen sólido que interactúa físicamente con Gazebo para el cálculo de las colisiones.
    - inertial: Establece la masa y su matriz de inercia, lo cual es vital para el cálculo dínamico.

- Las articulaciones o juntas (joints): Son los elementos cinemáticos que conectan los eslabones:

    - Utiliza juntas tipo revolute para conectar los eslabones principales. Esto porque giran sobre el eje Z y tienen restricciones físicas, con límites de posición entre -3.1416 y 3.1416 radianes, además de límites de velocidad y esfuerzo.
    - Se utilizan juntas tipo fixes para anclar la base al entorno y para colocar el efector final P al final del tercer eslabón.

3. Describir las relaciones del archivo xacro del sensor de la cámara.

El archivo camera_sensor.xacro tiene una doble relación: una mecánica con el robot y una de comunicación con el simulador Gazebo:

- Relación mecánica: Crea el cuerpo físico de la cámara y lo une al robot utilizando una junta fija. El eslabón padre es P, lo que significa que la cámara esta unida directamente en el efector final del brazo y se mueve con él.
- Relación óptica: Crea un eslabón auxiliar llamado camera_link_optical rotado -90 grados en los ejes Roll y Yaw. Esto adapta el modelos a los estándares de visión artificial, donde el eje Z debe apuntar hacia adelante (hacia la lente).
- Relación de simulación: Utiliza la etiqueta _gazebo reference="camera_link"_ para instanciar el plugin del sensorm configurando parámetros físicos de la cámara y establece que publicará las imágenes en el tópico de ROS 2 llamado _camera/image_raw_.

4. Describir el contenido del paquete scara_bringup y del archivo launch.

El paquete _scara_bringup_ funciona como el orquestador del proyecto. Su objetivo es juntar los elementos aislados y ejecutarlos de forma coordinada. Contiene entornos de simulación, las plantillas de visualización y los scripts de arranque.

El script _gz2_scara.launch.py_ automatiza el despliegue del sistema levantando simultáneamente cinco procesos fundamentales:

   1. _robot_state_publisher_node_: Procesa el archivo Xacro y publica la cinemática del robot en la res de ROS 2.
   2. _gz_sim_: Ejecuta la aplicación de Gazebo en segundo plano cargando el archivo del entorno virtual.
   3. _spaw_entity_: Hace aparecer el modelo del robot SCARA dentro del mundo virtual de Gazebo.
   4. _gz_ros_bridge_node_: Inicia el traductor de comunicaciones entre ROS 2 y Gazebo.
   5. _rviz_node_: Abre la interfaz gráfica de RViz2 con la configuración predefinida para visualizar la telerimetría, el modelo 3D y los sensores.

5 . Describir que hace el archivo bridge.

ROS 2 y Gazebo son ecosistemas independientes que utilizan protocolos de red distintos para comunicarse. El archivo bridge actúa como un traductor bidireccional en tiempo real.

Sus funciones principales son:

   - Tomar los tópicos generados en Gazebo y empaquetarlos como mensajes estándar que ROS 2 puede entender y visualizar.
   - Tomas las señales de control de ROS 2 y traducirlas al formato interno de Gazebo para que se muevan los motores virtuales.
   - Sincronozar el reloj de ambos sistemas, asegurando que ROS 2 y Gazebo compartan el mismo tiempo de simulación para evitar problemas de control.

# Previo de la Práctica 6

Se le solicita al alumno investigar y colocar la sigueinte información en esta sección:

- Investigar los simuladores que son compatibles con ROS 2.

Ross 2 cuenta con una arquitectura abierta que le permita comunicarse con diversos motores de simulación a través de paquetes de nodos dedicados. Los simuladores más utilizados y compatibles son:

   - Gazebo: Es el estándar de facto en la comunidad de ROS;  nos ofrece simulación de dinámicas físicas robustas e integración nativa de sensores.

   - Webots: Un entorno ágil y de código abierto muy eficiente para simulaciones rápidas de robots móviles y brazos robóticos sin consumir excesivos recursos de hardware.

   - Isaac Sim: Plataforma de alto rendimiento basada en omniverse. Utiliza aceleración por Hardware para simulaciones fotorrealistas y entrenamiento avanzado de Inteligencia artificial.

   - CoppeliaSim: Altamente versátil en la industria para modelado cinemático y dinámico, compatible mediante el plugin.


- Investigar cuales son las restricciones del simulador Gazebo 2.

   - Consumo elevado de recursos gráficos y de cómputo: El renderizado tridimensional y el cálculo dinámico de colisiones e inercias en tiempo real exigen un hardware robusto.

   - Discretización y aproximación de la física: Al basarse en solucionadores numéricos por pasos de tiempo discretos, pueden ocurrir errores de penetración de superficies o inestabilidad numérica si las matrices de inercia y fricción de los archivos _.xacro_ no están perfectamente calculadas. 

   - Sincronización de relojes: Depende críticamente del parámetro _use_sim_time_. Si el puente de comunicación experimenta latencia, el desfase entre el tiempo del simulador y el tiempo de procesamiento de ROS 2 puede romper la publicación de transformaciones 



## Resultados

En esta sección se presentan las evidencias de la ejecución del entorno de simulación. Al ejecutar el archivo descriptor de lanzamiento, se logró inicializar de manera síncrona el motor de física de Gazebo junto con la interfaz de visualización RViz2, además de establecer el puente de comunicación.

![Gazebo](Images/Gazebo.png)
*Figura 1. Entorno virtual en Gazebo con el modelo del robot SCARA instanciado.*

![RViz2](Images/Ros2.png)
*Figura 2. Interfaz de RViz2 reflejando las transformaciones cinemáticas (TF) del robot.*

![Topic List](Images/Ros2_TopicList.png)
*Figura 3. Listado de tópicos activos de ROS 2 puenteados desde el simulador Gazebo.*

![Topic Camera](Images/Ros2_TopicCamera.png)
*Figura 4. Eco de datos del tópico de la cámara demostrando la adquisición de imágenes en tiempo real.*

[![Simulación de trayectoria lineal SCARA](https://img.youtube.com/vi/F9YmN5ys3Bc/hqdefault.jpg)](https://youtu.be/F9YmN5ys3Bc)

*Figura 5. Simulación de trayectoria lineal del robot SCARA en Gazebo. (Haz clic en la imagen para reproducir).*

## Conclusiones

Durante la realización de esta práctica, cumplí exitosamente con el objetivo de integrar y coordinar un entorno de simulación completo para un robot Industrial tipo SCARA empleando ROS 2, Jazzy y Gazebo. El desarrollo permitió comprender el valor de la arquitectura modular basada en archivos Xacro;  la separación clara entre el modelo geométrico, los controladores dinámicos y los sensores periféricos demostró ser una práctica de ingeniería eficiente que facilita el mantenimiento y la estabilidad del sistema sin alterar la cinemática base del manipulador.

De la misma forma, el despliegue mediante el _script gz2_scara.launch.py_ evidenció la necesidad crítica de implementar herramientas de traducción bidireccional en tiempo real, como el nodo _ros_gz_bridge_ .  Se comprobó que el correcto funcionamiento de la percepción del robot y del control posicional de las juntas depende enteramente de la sincronización de los relojes virtuales del sistema. Finalmente, la experiencia práctica nos demostró la importancia de optimizar la asignación de recursos de Hardware al trabajar con entornos virtuales y Software de simulación robótica de alta demanda, lo cual garantiza la estabilidad operativa del sistema de desarrollo de software. 


## Bibliografía 

[1] J. M. Gómez de Gabriel, A. Mandow y J. Martínez, Programación de Robots con ROS, 1ª ed. Málaga, España: Universidad de Málaga, 2021.

[2] A. Guerrero de la Peña, "Desarrollo de un entorno de simulación robótica modular utilizando ROS 2 y Gazebo," tesis de licenciatura, Fac. de Ingeniería, Universidad Nacional Autónoma de México, CDMX, México, 2024.

[3] F. N. Martínez Torres, "Integración de sensores virtuales y control de movimiento en entornos industriales mediante el Sistema Operativo para Robots (ROS)," Revista Iberoamericana de Automática e Informática Industrial, vol. 20, n.º 3, pp. 245-256, jul. 2023. doi: 10.4995/riai.2023.18452.

[4] R. Morales, "Guía práctica de simulación en robótica: Configuración de sensores y entornos en Gazebo," Departamento de Automática, Universidad de Alcalá, Madrid, España, Repositorio Institucional, 2025. [En línea]. Disponible en: http://hdl.handle.net/10017/54321. [Consultado: 21-may-2026].

[5] E. S. Castro y L. V. Benítez, "Estudio comparativo de puentes de comunicación y middleware para arquitecturas distribuidas en ROS 2," Boletín Técnico de Tecnologías de la Información, vol. 15, n.º 2, pp. 89-102, feb. 2025.